# Dhara Pipeline Dry Run

This notebook replicates the Dhara backend flow end-to-end, using the **real** backend modules
(`extractor.py`, `metadata_excel.py`, `catalogue_matching.py`, `catalogue.py`) so what you see here
is exactly what the API does — just called directly, with the output of each stage printed inline.

**Stages covered (this notebook stops right before column classification):**

1. Load a sample dataset workbook (a real government-statistical-style Excel upload)
2. **Table extraction** — `TableExtractor.extract_from_file` (grid building → block detection → structure inference → DDI table IDs)
3. **KYDS (Know Your Dataset) form** — simulate a form submission and persist it
4. **Metadata Excel parsing** — parse a `catalogue_summary` / `dataset_inventory_list` workbook (`metadata_excel.py`)
5. **Table ↔ metadata matching / grouping** — `catalogue_matching.py`
6. **Metadata filling** — assemble the metadata group record that would be pushed to the catalogue (`catalogue.push_to_catalogue`)

⏸️ **Stops here** — next stage (not in this notebook) is per-table **column classification**.

---

### Notes on running this
- Run from the `backend/` directory (or adjust `sys.path` below) so `extractor`, `metadata_excel`, etc. import cleanly.
- Use the project's `.venv` as the kernel (`backend/.venv`) — it already has `openpyxl`, `anthropic`, `psycopg2`.
- LLM calls are **skipped by default** (`SKIP_LLM = True` below) so this runs with no API key, deterministically, using the same heuristic fallback the backend uses when a user has no LLM key configured. Flip the flag if you want to exercise the real Claude-backed extraction/enrichment paths.
- Postgres push is **optional** — gated behind `RUN_DB_PUSH`. If `DATABASE_URL` isn't reachable, the notebook still shows the exact payload that *would* be written.
- **Input format**: the sample workbook in `sample_data/` follows the same layout your real files use — a `TABLE: <code>` marker row, a title row, one or two-level merged column headers, a data body (often ending in an `ALL` total row), and sheets that pack an URBAN block immediately followed by a RURAL block. Drop your own workbook into `sample_data/` and point `SAMPLE_XLSX_PATH` at it to dry-run a different file.

In [23]:
import sys, os, json
from pathlib import Path

BACKEND_DIR = Path.cwd() if (Path.cwd() / "extractor.py").exists() else Path.cwd() / "backend"
sys.path.insert(0, str(BACKEND_DIR))
os.chdir(BACKEND_DIR)
print("Using backend dir:", BACKEND_DIR)

from dotenv import load_dotenv
load_dotenv()

SKIP_LLM = False     # set False to use real Claude calls (needs ANTHROPIC_API_KEY)
RUN_DB_PUSH = True  # set True to actually write to Postgres (needs DATABASE_URL reachable)

from extractor import TableExtractor
import catalogue as cat
from metadata_llm import extract_excel_facts, generate_metadata_with_llm, parse_llm_metadata_output

print("Modules loaded OK")

Using backend dir: /Users/sriramrahul/Desktop/dhara-toolkit-own/dhara-poc/backend
Modules loaded OK


## Stage 0 — Load a sample dataset workbook

Dhara ingests government-style statistical Excel workbooks: a `TABLE: <code>` marker row, a title/description,
one or two-level merged column headers (e.g. `INSTITUTIONAL` spanning `MALE/FEMALE/OTHER/TOTAL`), a data body
(often ending in an `ALL` total row), and sheets that frequently pack two sub-tables back to back — an URBAN
block immediately followed by a RURAL block on the same sheet.

This notebook loads a **real** sample workbook of that exact shape
(`sample_data/Infant_Mother_Death_D12-D18.xlsx` — Infant & Mother Death tables D-12 through D-18, 5 sheets,
URBAN/RURAL split per sheet) so the extraction stage below runs against actual field data rather than a
synthetic stand-in. Swap in any similarly-shaped workbook by dropping it into `sample_data/` and updating
`SAMPLE_XLSX_PATH`.

In [24]:
import openpyxl

SAMPLE_XLSX_PATH = BACKEND_DIR / "sample_data" / "Infant_Mother_Death_D12-D18.xlsx"

# SAMPLE_XLSX_PATH = BACKEND_DIR / "sample_data" / "Still Birth Table-2024.xlsx"

sample_filename = SAMPLE_XLSX_PATH.name
sample_bytes = SAMPLE_XLSX_PATH.read_bytes()

wb_preview = openpyxl.load_workbook(SAMPLE_XLSX_PATH, data_only=True)
print(f"Loaded {sample_filename} ({len(sample_bytes)} bytes)")
print("Sheets:", wb_preview.sheetnames)

Loaded Infant_Mother_Death_D12-D18.xlsx (253856 bytes)
Sheets: ['D-12 & D-13', 'D-14', 'D-15', 'D-16', 'D-18']


## Stage 1 — Table extraction

`TableExtractor.extract_from_file` (the same code the `/api/extract` and `/api/catalogue/batch-extract`
endpoints call):

1. Loads the workbook, builds a merge-resolved "filled grid" per sheet
2. Finds table blocks (via `TABLE:` markers, or blank-row fallback)
3. Strips title/description rows, infers header/skip rows and column names
   (heuristically here since `SKIP_LLM=True`; the direct-LLM strategy would replace `_heuristic_structure`)
4. Builds row dicts, captures raw header rows / footnotes
5. Generates a DDI-format table ID and dedupes IDs/titles across the whole batch

In [26]:
extractor = TableExtractor(api_key=None, skip_llm=False)
tables = extractor.extract_from_file(sample_bytes, sample_filename)

for t in tables:
    t["source_file"] = sample_filename

print(f"Extracted {len(tables)} tables\n")
for t in tables:
    print(f"- id={t['id']}")
    print(f"    title       : {t['title']}")
    print(f"    description : {t['description']}")
    print(f"    sheet       : {t['sheet']}")
    print(f"    columns     : {t['columns']}")
    print(f"    row_count   : {t['row_count']}")
    print(f"    sample row  : {t['rows'][0] if t['rows'] else None}")
    print()

[DEBUG] sheet='D-12 & D-13'  grid_rows=18  blocks=[(0, 7), (10, 17)][DEBUG] sheet='D-14'  grid_rows=8  blocks=[(0, 7)]

[DEBUG] sheet='D-15'  grid_rows=18  blocks=[(0, 7), (10, 17)]
[DEBUG] sheet='D-16'  grid_rows=18  blocks=[(0, 7), (10, 17)]
[DEBUG] sheet='D-18'  grid_rows=29  blocks=[(0, 13), (15, 28)]
Structure analysis failed (Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011CeagCTCTrNtD8D5xP8oiu'}), using heuristic
Structure analysis failed (Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011CeagCTFC3b2kwWVrDJbfx'}), using heuristic
Structure analysis failed (Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011CeagCTEwxuHUjyLSyGkRY'}), using heuristic
Structure analysis failed (Error code: 401 - {'type': 'error', 'error': {'type': 'authe

In [22]:
# Full structure of one extracted table, for inspection (D-12/D-13, urban block)
print(json.dumps(tables[0], indent=2, default=str))

{
  "id": "DDI_DEL_DES_VS_D12_URBAN_2024_V1",
  "title": "TABLE: D-12 & D-13",
  "description": "INFANT DEATHS BY PLACE OF OCCURRENCE, DISTRICTS (URBAN)",
  "sheet": "D-12 & D-13",
  "filename": "Infant_Mother_Death_D12-D18.xlsx",
  "columns": [
    "SL. NO.",
    "DISTRICT",
    "INSTITUTIONAL",
    "INSTITUTIONAL_1",
    "INSTITUTIONAL_2",
    "INSTITUTIONAL_3",
    "DOMICILIARY",
    "DOMICILIARY_1",
    "DOMICILIARY_2",
    "DOMICILIARY_3",
    "ALL",
    "ALL_1",
    "ALL_2",
    "ALL_3"
  ],
  "rows": [
    {
      "SL. NO.": 1.0,
      "DISTRICT": "MCD",
      "INSTITUTIONAL": 2241.0,
      "INSTITUTIONAL_1": 1624.0,
      "INSTITUTIONAL_2": 6.0,
      "INSTITUTIONAL_3": 3871,
      "DOMICILIARY": 14.0,
      "DOMICILIARY_1": 20.0,
      "DOMICILIARY_2": 0.0,
      "DOMICILIARY_3": 34,
      "ALL": 2255,
      "ALL_1": 1644,
      "ALL_2": 6,
      "ALL_3": 3905
    },
    {
      "SL. NO.": 2.0,
      "DISTRICT": "NDMC   ",
      "INSTITUTIONAL": 703.0,
      "INSTITUTIONAL_1":

### Optional: LLM-based category metadata extraction

`extract_category_metadata` (used by `/api/table-metadata`) asks the LLM to identify categorical
dimensions (Area Type, Gender, Age Group, …) from a table's raw headers + sample rows. It's a no-op
when `skip_llm=True` (returns `[]`) — shown here for completeness of the flow.

In [19]:
categories = extractor.extract_category_metadata(
    title=tables[0]["title"],
    description=tables[0]["description"],
    raw_header_rows=tables[0]["raw_header_rows"],
    columns=tables[0]["columns"],
    sample_rows=tables[0]["rows"],
    raw_notes=tables[0]["raw_notes"],
)
print("Categories (empty because SKIP_LLM=True):", categories)

Categories (empty because SKIP_LLM=True): []


## Stage 2 — KYDS (Know Your Dataset) form

Mirrors `POST /api/kyds`: a free-form `responses` object plus a `user` block, persisted via
`catalogue.save_kyds_entry`. This is independent of the extraction pipeline — it's a
dataset-intake questionnaire captured alongside the upload.

In [5]:
# kyds_user = {"email": "aparajita@peopleplus.ai", "name": "Aparajita", "dept": "DES"}
# kyds_responses = {
#     "dataset_purpose": "Track live births and deaths by district for health planning",
#     "collection_method": "Administrative registration records",
#     "update_frequency": "Annual",
#     "known_limitations": "Provisional figures pending revision",
#     "sensitive_data": False,
# }

# print("KYDS submission payload:")
# print(json.dumps({"user": kyds_user, "responses": kyds_responses}, indent=2))

kyds_user = {
    "email": "aparajita@peopleplus.ai",
    "name": "Aparajita",
    "dept": "DES",
}

kyds_responses = {
    # 1. Modality
    "modality": ["Structured data"],
    "other_modality_describe": "",
    "specific_formats": "Excel (.xlsx)",

    # 2. Sensitivity and classification
    "dpdp_tiers": [],
    "special_category_subtypes": [],
    "degree_per_tier": "",

    "org_classification": ["Internal"],
    "national_classification": [],

    # 3. Access level
    "access_level": ["Open"],
    "more_open_subset": "",
    "restricted_sharing_partner": "",
    "embargoed_release": "",

    # 4. Granularity
    "granularity": ["District"],
    "restricted_to_sub_population": "No",
    "linked_persistent_id": "No",
    "longitudinal": "No",

    # 5. Update frequency and retention
    "update_frequency": ["Annual"],
    "retention": [],
    "retention_citation": "",

    # 6. Storage
    "storage": ["Online"],

    # 7. Notes and lawful basis
    "restricted_lawful_basis": "",
    "special_category_basis": "",
    "dpia": "No",
    "notes": "",

    # # Existing fields
    # "dataset_purpose": "Track live births and deaths by district for health planning",
    # "collection_method": "Administrative registration records",
    # "known_limitations": "Provisional figures pending revision",
    "sensitive_data": False,
}

print("KYDS submission payload:")
print(json.dumps({
    "user": kyds_user,
    "responses": kyds_responses
}, indent=2))

KYDS submission payload:
{
  "user": {
    "email": "aparajita@peopleplus.ai",
    "name": "Aparajita",
    "dept": "DES"
  },
  "responses": {
    "modality": [
      "Structured data"
    ],
    "other_modality_describe": "",
    "specific_formats": "Excel (.xlsx)",
    "dpdp_tiers": [],
    "special_category_subtypes": [],
    "degree_per_tier": "",
    "org_classification": [
      "Internal"
    ],
    "national_classification": [],
    "access_level": [
      "Open"
    ],
    "more_open_subset": "",
    "restricted_sharing_partner": "",
    "embargoed_release": "",
    "granularity": [
      "District"
    ],
    "restricted_to_sub_population": "No",
    "linked_persistent_id": "No",
    "longitudinal": "No",
    "update_frequency": [
      "Annual"
    ],
    "retention": [],
    "retention_citation": "",
    "storage": [
      "Online"
    ],
    "restricted_lawful_basis": "",
    "special_category_basis": "",
    "dpia": "No",
    "notes": "",
    "sensitive_data": false
  }


In [6]:
kyds_entry_id = None
if RUN_DB_PUSH:
    conn = cat.get_connection()
    cat.init_schema(conn)
    kyds_entry_id = cat.save_kyds_entry(conn, kyds_responses, kyds_user)
    conn.close()
    print(f"Saved KYDS entry, id={kyds_entry_id}")
else:
    print("RUN_DB_PUSH=False — skipping actual DB write. This is the row that *would* be inserted "
          "into kyds_entries:")
    print(json.dumps({"user_email": kyds_user["email"], "user_name": kyds_user["name"],
                       "user_dept": kyds_user["dept"], "responses": kyds_responses}, indent=2))

Saved KYDS entry, id=16


## Stage 3 — Automatic Grouping

Tables are grouped automatically by extracting common patterns from their IDs and descriptions.
Tables with the same sheet + matching ID prefixes (ignoring URBAN/RURAL/version suffixes) are grouped together.
This eliminates the need for an external metadata workbook's inventory sheet.

In [17]:
import re

def auto_group_tables(tables):
    """
    Group tables automatically by extracting ID patterns.
    
    Removes URBAN/RURAL/version suffixes to find common base IDs.
    Tables from the same sheet with matching base IDs are grouped.
    """
    groups_dict = {}
    
    for table in tables:
        table_id = table["id"]
        sheet = table["sheet"]
        
        # Extract base ID by removing URBAN/RURAL and version suffixes
        # Pattern: DDI_DEL_DES_VS_S1_URBAN_2024_V1 -> base is S1 or core prefix
        base_match = re.match(r"(DDI_[A-Z]+_[A-Z]+_[A-Z]+_[A-Z0-9]+)", table_id)
        if base_match:
            base_prefix = base_match.group(1)
        else:
            base_prefix = table_id
        
        # Remove URBAN/RURAL to get the group key
        group_key_base = re.sub(r"_(URBAN|RURAL).*", "", base_prefix)
        group_key = (sheet, group_key_base)
        
        if group_key not in groups_dict:
            groups_dict[group_key] = {
                "sheet": sheet,
                "base_id": group_key_base,
                "table_ids": [],
                "tables": [],
            }
        
        groups_dict[group_key]["table_ids"].append(table_id)
        groups_dict[group_key]["tables"].append(table)
    
    return list(groups_dict.values())

# Auto-group extracted tables
groups = auto_group_tables(tables)

print(f"Auto-grouped {len(tables)} tables into {len(groups)} group(s)\\n")
for i, g in enumerate(groups):
    print(f"Group {i+1}: sheet={g['sheet']}, base_id={g['base_id']}")
    print(f"  Table IDs: {g['table_ids']}")
    print()

Auto-grouped 9 tables into 5 group(s)\n
Group 1: sheet=D-12 & D-13, base_id=DDI_DEL_DES_VS_D12
  Table IDs: ['DDI_DEL_DES_VS_D12_URBAN_2024_V1', 'DDI_DEL_DES_VS_D12_RURAL_2024_V1']

Group 2: sheet=D-14, base_id=DDI_DEL_DES_VS_D14
  Table IDs: ['DDI_DEL_DES_VS_D14_2024_V1']

Group 3: sheet=D-15, base_id=DDI_DEL_DES_VS_D15
  Table IDs: ['DDI_DEL_DES_VS_D15_URBAN_2024_V1', 'DDI_DEL_DES_VS_D15_RURAL_2024_V1']

Group 4: sheet=D-16, base_id=DDI_DEL_DES_VS_D16
  Table IDs: ['DDI_DEL_DES_VS_D16_URBAN_2024_V1', 'DDI_DEL_DES_VS_D16_RURAL_2024_V1']

Group 5: sheet=D-18, base_id=DDI_DEL_DES_VS_D18
  Table IDs: ['DDI_DEL_DES_VS_D18_URBAN_2024_V1', 'DDI_DEL_DES_VS_D18_RURAL_2024_V1']



## Stage 3 — Metadata Excel parsing

The DES metadata workbook has a fixed `catalogue_summary` sheet (group-level fields: product,
category, geography, frequency, …) plus a `dataset_inventory_list` sheet (per-table IDs +
descriptions) and optional classification / concept sheets.

`parse_catalogue_summary` powers the "Create Metadata" form prefill
(`POST /api/catalogue/parse-metadata-excel`); `parse_metadata_workbook` does the full parse used by
batch matching.

In [18]:
def build_sample_metadata_workbook() -> bytes:
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "catalogue_summary"
    ws.append(["product", "category", "geography", "frequency", "time_period",
               "data_source", "description", "last_updated_date", "future_release",
               "key_statistics", "remarks"])
    ws.append(["Infant & Mother Death Statistics", "Health", "Delhi", "Annual", "2024",
               "Civil Registration System", "Infant and maternal deaths registered across districts, "
               "by place of occurrence, age, cause, and occupation",
               "January, 2025", "January, 2026", "6,866 infant deaths recorded",
               "Provisional figures, subject to revision"])

    inv = wb.create_sheet("dataset_inventory_list")
    inv.append(["sl#", "unique dataset id", "table id", "dataset short description", "dataset long description"])
    # D-12/D-13: exact-ID match (extractor's own generated IDs, copied verbatim)
    inv.append([1, "DDI_DEL_DES_VS_D12_URBAN_2024_V1", "D-12",
                "Infant deaths by place of occurrence, urban",
                "District-wise infant deaths split by institutional vs domiciliary occurrence, urban areas"])
    inv.append([2, "DDI_DEL_DES_VS_D12_RURAL_2024_V1", "D-12",
                "Infant deaths by place of occurrence, rural",
                "District-wise infant deaths split by institutional vs domiciliary occurrence, rural areas"])
    # D-14: table-code match (single sheet, no urban/rural split -> unambiguous code match)
    inv.append([3, "DDI_DEL_DES_VS_D14_2024_V1", "D-14",
                "Infant deaths by age and sex",
                "Infant deaths broken down by age band (<7 days, 7-28 days, 28 days-1 year) and sex"])
    # D-15/D-16: code match with >1 candidate per code, disambiguated by keyword (urban/rural)
    inv.append([4, "DDI_DEL_DES_VS_D15_URBAN_ALL_2024_V1", "D-15",
                "Pregnancy related deaths by age and cause, medically certified, urban",
                "Pregnancy-related deaths by maternal age group and cause of death for medically "
                "certified deaths, urban areas"])
    inv.append([5, "DDI_DEL_DES_VS_D15_RURAL_ALL_2024_V1", "D-15",
                "Pregnancy related deaths by age and cause, medically certified, rural",
                "Pregnancy-related deaths by maternal age group and cause of death for medically "
                "certified deaths, rural areas"])
    inv.append([6, "DDI_DEL_DES_VS_D16_URBAN_ALL_2024_V1", "D-16",
                "Pregnancy related deaths by age and cause, certified or not, urban",
                "Pregnancy-related deaths by maternal age group and cause of death for medically "
                "certified or not deaths, urban areas"])
    inv.append([7, "DDI_DEL_DES_VS_D16_RURAL_ALL_2024_V1", "D-16",
                "Pregnancy related deaths by age and cause, certified or not, rural",
                "Pregnancy-related deaths by maternal age group and cause of death for medically "
                "certified or not deaths, rural areas"])
    # D-18: left out of the inventory on purpose, to also show the "no metadata match" path
    # (D-18 tables fall back into their own group with empty catalogue fields)

    import io
    buf = io.BytesIO()
    wb.save(buf)
    return buf.getvalue()

metadata_bytes = build_sample_metadata_workbook()
metadata_filename = "sample_metadata.xlsx"

prefill_fields = parse_catalogue_summary(metadata_bytes)
print("Create Metadata form prefill (from catalogue_summary):")
print(json.dumps(prefill_fields, indent=2))

NameError: name 'parse_catalogue_summary' is not defined

In [11]:
metadata_workbook = parse_metadata_workbook(metadata_bytes, metadata_filename)
print("Full parsed metadata workbook:")
print(json.dumps(metadata_workbook, indent=2, default=str))

Full parsed metadata workbook:
{
  "file_name": "sample_metadata.xlsx",
  "summary": {
    "title": "Infant & Mother Death Statistics",
    "product": "Infant & Mother Death Statistics",
    "category": "Health",
    "geography": "Delhi",
    "frequency": "Annual",
    "time_period": "2024",
    "data_source": "Civil Registration System",
    "description": "Infant and maternal deaths registered across districts, by place of occurrence, age, cause, and occupation",
    "last_updated": "January, 2025",
    "future_release": "January, 2026",
    "key_statistics": "6,866 infant deaths recorded",
    "remarks": "Provisional figures, subject to revision"
  },
  "inventory": [
    {
      "sl_no": 1,
      "unique_dataset_id": "DDI_DEL_DES_VS_D12_URBAN_2024_V1",
      "table_id": "D-12",
      "short_description": "Infant deaths by place of occurrence, urban",
      "long_description": "District-wise infant deaths split by institutional vs domiciliary occurrence, urban areas"
    },
    {
  

## Stage 4 — Matching extracted tables to metadata, and grouping

`match_tables_to_metadata` (used by `/api/catalogue/batch-match` and `/api/group-tables`) tries, in
order: exact ID match → ID-stem match (tolerant of year/version typos) → table-code match
(disambiguated by a distinguishing keyword when a code maps to multiple inventory rows).

`match_result_to_push_groups` then flattens that into the simpler `{name, table_ids}` shape used to
drive the single-file push UI.

In [11]:
match_result = match_tables_to_metadata(tables, [metadata_workbook])

print(f"Groups: {len(match_result['groups'])}")
for g in match_result["groups"]:
    print(f"\nGroup (workbook_index={g['workbook_index']}, file={g['file_name']}):")
    print(f"  metadata.product = {g['metadata'].get('product')}")
    for mt in g["matched_tables"]:
        print(f"  - table {mt['table']['id']!r}  confidence={mt['confidence']}"
              f"  matched_inventory_id={ (mt['inventory_item'] or {}).get('unique_dataset_id') }")

print("\nUnmatched tables:", [u["table"]["id"] for u in match_result["unmatched_tables"]])
print("Unmatched inventory rows:",
      [i["inventory_item"]["unique_dataset_id"] for i in match_result["unmatched_inventory"]])

NameError: name 'match_tables_to_metadata' is not defined

In [10]:
push_groups = match_result_to_push_groups(match_result)
print("Push-ready groups ({name, table_ids}):")
print(json.dumps(push_groups, indent=2))

Push-ready groups ({name, table_ids}):
[
  {
    "name": "sample_metadata.xlsx",
    "table_ids": [
      "DDI_DEL_DES_VS_D12_URBAN_2024_V1",
      "DDI_DEL_DES_VS_D12_RURAL_2024_V1",
      "DDI_DEL_DES_VS_D14_2024_V1",
      "DDI_DEL_DES_VS_D15_URBAN_2024_V1",
      "DDI_DEL_DES_VS_D15_RURAL_2024_V1",
      "DDI_DEL_DES_VS_D16_URBAN_2024_V1",
      "DDI_DEL_DES_VS_D16_RURAL_2024_V1",
      "DDI_DEL_DES_VS_D18_URBAN_2024_V1",
      "DDI_DEL_DES_VS_D18_RURAL_2024_V1"
    ]
  }
]


## Stage 4 — LLM Metadata Creation per Group

Once tables are grouped, the LLM generates metadata for each group using:

1. **Excel-derived facts** — `metadata_llm.extract_excel_facts` extracts sheet names, table boundaries, headers, column names, sample rows, geography, and units from the workbook
2. **KYDS responses** — human-provided form data from Stage 2

The LLM uses both to populate product, category, geography, frequency, title, description, etc. for the group.

In [9]:
def generate_metadata_per_group(groups, excel_bytes, filename, kyds_responses, skip_llm=False):
    """
    Generate LLM metadata for each group.
    
    Args:
        groups: list of group dicts (from auto_group_tables)
        excel_bytes: workbook bytes
        filename: workbook filename
        kyds_responses: KYDS form responses
        skip_llm: if True, use heuristic fallback
    
    Returns:
        dict mapping group_key -> llm_filled_metadata
    """
    excel_facts = extract_excel_facts(excel_bytes, filename)
    group_metadata = {}
    
    for group in groups:
        group_key = (group["sheet"], group["base_id"])
        
        if not skip_llm:
            llm_output = generate_metadata_with_llm(
                excel_facts,
                kyds=kyds_responses,
                api_key=os.getenv("OPENAI_API_KEY"),
            )
            llm_filled = parse_llm_metadata_output(llm_output)
        else:
            # Heuristic: derive from table descriptions
            descriptions = [t.get("description", "") for t in group["tables"]]
            desc_str = " ".join(descriptions)
            
            llm_filled = {
                "title": group["base_id"],
                "product": "Health Statistics",
                "category": "Health",
                "geography": ["District", "Urban", "Rural"] if "URBAN" in desc_str or "RURAL" in desc_str else ["District"],
                "frequency": "Annual",
                "time_period": None,
                "data_source": None,
                "description": " | ".join(descriptions[:2]),
                "last_updated": None,
                "future_release": None,
                "key_statistics": None,
                "remarks": None,
            }
        
        group_metadata[group_key] = llm_filled
    
    return group_metadata

# Generate metadata for each group
group_metadata = generate_metadata_per_group(groups, sample_bytes, sample_filename, kyds_responses, skip_llm=SKIP_LLM)


In [10]:
print("Generated metadata for each group:\\n")
for group in groups:
    group_key = (group["sheet"], group["base_id"])
    meta = group_metadata[group_key]
    print(f"Group: sheet={group['sheet']}, base_id={group['base_id']}")
    print(f"  Tables: {group['table_ids']}")
    print(f"  Metadata: product={meta.get('product')}, category={meta.get('category')}, geography={meta.get('geography')}, description={meta.get('description')}")
    print()

Generated metadata for each group:\n
Group: sheet=D-12 & D-13, base_id=DDI_DEL_DES_VS_D12
  Tables: ['DDI_DEL_DES_VS_D12_URBAN_2024_V1', 'DDI_DEL_DES_VS_D12_RURAL_2024_V1']
  Metadata: product=Infant and Maternal Mortality Dataset, category=Health Statistics, geography=District, description=This dataset contains statistics on infant deaths by place of occurrence and district in both urban and rural settings, as well as pregnancy-related deaths categorized by age group and occupation of the deceased mothers.

Group: sheet=D-14, base_id=DDI_DEL_DES_VS_D14
  Tables: ['DDI_DEL_DES_VS_D14_2024_V1']
  Metadata: product=Infant and Mother Mortality Reports, category=Health Statistics, geography=District, description=This dataset includes statistics on infant deaths by place of occurrence, pregnancy-related deaths segmented by cause, age, and geography (urban and rural). The statistics include detailed data breakdowns by sex and age group for both infant and mother mortality.

Group: sheet=D-15

In [17]:
for group in groups:
    group_key = (group["sheet"], group["base_id"])
    meta = group_metadata[group_key]
    group_tables = group["tables"]
    
    # Placeholder enrichment (real enrichment = extractor.enrich_for_catalogue, the next stage)
    placeholder_enriched = [
        {
            "short_description": t.get("description") or t.get("title", ""),
            "long_description": t.get("description") or t.get("title", ""),
            "units": "Count",
            "classifications": {},   # <-- column classification would populate this
            "age_column_keys": {},
        }
        for t in group_tables
    ]
    
    print(f"\\n{'='*60}")
    print(f"Pushing group: {meta.get('product')} > {meta.get('category')}")
    print(f"  Tables: {group['table_ids']}")
    print(f"  Metadata fields:")
    for field, value in meta.items():
        if value is not None:
            print(f"    {field:16}: {value}")
    
    if RUN_DB_PUSH:
        conn = cat.get_connection()
        cat.init_schema(conn)
        result = cat.push_to_catalogue(
            conn, group_tables, placeholder_enriched, "new", None,
            meta.get("title") or meta.get("product"), meta.get("description"),
            meta.get("product"), meta.get("category"), 
            str(meta.get("geography")) if meta.get("geography") else None,
            meta.get("frequency"), meta.get("time_period"), meta.get("data_source"),
            meta.get("last_updated"), meta.get("future_release"),
            meta.get("key_statistics"), meta.get("remarks"), None,
        )
        conn.close()
        print(f"\\n  Result: metadata_id={result.get('metadata_id')}, {result.get('tables_pushed')} tables pushed")
    else:
        print(f"\\n  RUN_DB_PUSH=False — would insert:")
        print(json.dumps({
            "title": meta.get("title") or meta.get("product"),
            "product": meta.get("product"),
            "category": meta.get("category"),
            "table_ids": group["table_ids"],
        }, indent=4))

print(f"\\n{'='*60}")
print(f"\\nPipeline complete: {len(groups)} group(s), {len(tables)} table(s) total")

\n============================================================
Pushing group: Infant and Mother Death Statistics > Health Statistics
  Tables: ['DDI_DEL_DES_VS_D12_URBAN_2024_V1', 'DDI_DEL_DES_VS_D12_RURAL_2024_V1']
  Metadata fields:
    title           : Infant and Mother Death Statistics in Urban and Rural Districts
    product         : Infant and Mother Death Statistics
    category        : Health Statistics
    geography       : District
    frequency       : Annual
    description     : The dataset provides detailed statistics on infant deaths by place of occurrence (urban and rural) and districts, as well as pregnancy-related deaths by age and cause in urban and rural settings. It includes data on both institutional and domiciliary deaths segmented by gender, age, and occupation.
    key_statistics  : {'urban_infant_deaths': 5225, 'rural_infant_deaths': 1641, 'total_pregnancy_related_deaths_urban': 79, 'total_pregnancy_related_deaths_rural': 57}
    remarks         : Data is p

UniqueViolation: duplicate key value violates unique constraint "datasets_pkey"
DETAIL:  Key (dataset_id)=(DDI_DEL_DES_VS_D12_URBAN_2024_V1) already exists.


## Stage 5 — Metadata Filling and Push to Catalogue

Now we push each group's metadata to the catalogue via `push_to_catalogue`.
This creates one `metadata_groups` row (group-level metadata) and one `datasets` row per table in the group.

Per-table enrichment (classifications, units, age_column_keys) is normally populated by
`extractor.enrich_for_catalogue` (column classification) — that's the next stage and is
intentionally excluded here. We use placeholder enrichment to show the exact shape the push expects.

In [13]:
from metadata_llm import extract_excel_facts, generate_metadata_with_llm

excel_facts = extract_excel_facts(sample_bytes, sample_filename)

print(f"Sheets processed: {len(excel_facts['sheets'])}")
for sheet in excel_facts["sheets"]:
    print(f"\nSheet: {sheet['sheet_name']}  ({len(sheet['tables'])} table block(s))")
    for tbl in sheet["tables"]:
        print(f"  - boundaries={tbl['table_boundaries']}  identifiers={tbl['table_identifiers']}"
              f"  multi_row_header={tbl['multi_row_header']}  geography={tbl['geography']}"
              f"  units={tbl['units']}  periods={tbl['periods']}")

Sheets processed: 5

Sheet: D-12 & D-13  (2 table block(s))
  - boundaries={'start_row': 0, 'end_row': 7, 'n_rows': 8, 'n_cols': 14}  identifiers=['D-12', 'D-13']  multi_row_header=True  geography=['DISTRICT', 'URBAN']  units=[]  periods={'years': [], 'year_ranges': []}
  - boundaries={'start_row': 10, 'end_row': 17, 'n_rows': 8, 'n_cols': 14}  identifiers=['D-12', 'D-13']  multi_row_header=True  geography=['DISTRICT', 'RURAL']  units=[]  periods={'years': [], 'year_ranges': []}

Sheet: D-14  (1 table block(s))
  - boundaries={'start_row': 0, 'end_row': 7, 'n_rows': 8, 'n_cols': 14}  identifiers=['D-14']  multi_row_header=True  geography=['RURAL', 'URBAN']  units=[]  periods={'years': ['1901'], 'year_ranges': []}

Sheet: D-15  (2 table block(s))
  - boundaries={'start_row': 0, 'end_row': 7, 'n_rows': 8, 'n_cols': 12}  identifiers=['D-15']  multi_row_header=True  geography=['URBAN']  units=[]  periods={'years': [], 'year_ranges': []}
  - boundaries={'start_row': 10, 'end_row': 17, 'n_ro

In [14]:
# Full excel_facts structure, for inspection
print(json.dumps(excel_facts, indent=2, default=str))

{
  "filename": "Infant_Mother_Death_D12-D18.xlsx",
  "sheets": [
    {
      "sheet_name": "D-12 & D-13",
      "tables": [
        {
          "sheet_name": "D-12 & D-13",
          "title_context": "TABLE: D-12 & D-13",
          "description": "INFANT DEATHS BY PLACE OF OCCURRENCE, DISTRICTS (URBAN)",
          "table_name": "D-12 & D-13 / TABLE: D-12 & D-13",
          "table_boundaries": {
            "start_row": 0,
            "end_row": 7,
            "n_rows": 8,
            "n_cols": 14
          },
          "table_identifiers": [
            "D-12",
            "D-13"
          ],
          "raw_header_rows": [
            [
              "SL. NO.",
              "DISTRICT",
              "INSTITUTIONAL",
              "INSTITUTIONAL",
              "INSTITUTIONAL",
              "INSTITUTIONAL",
              "DOMICILIARY",
              "DOMICILIARY",
              "DOMICILIARY",
              "DOMICILIARY",
              "ALL",
              "ALL",
              "ALL",


In [15]:
from metadata_llm import prepare_kyds_for_llm

if not SKIP_LLM:
    llm_metadata = generate_metadata_with_llm(
        excel_facts,
        kyds=kyds_responses,
        api_key=os.getenv("OPENAI_API_KEY"),
    )
    print(llm_metadata)
else:
    kyds_llm = prepare_kyds_for_llm(kyds_responses)
    print("SKIP_LLM=True — not calling OpenAI. Payload that would be sent (excel_facts + trimmed kyds):")
    print(json.dumps({
        "excel_facts": excel_facts,
        "dataset_context": kyds_llm,
    }, indent=2, default=str)[:2000], "...")

```json
{
  "title": "Infant and Mother Deaths by Geography and Age",
  "product": "Infant and Mother Death Statistics Dataset",
  "category": null,
  "geography": ["Urban", "Rural"],
  "frequency": "Annual",
  "time_period": null,
  "data_source": null,
  "description": "This dataset provides information on infant deaths categorized by place of occurrence, age, and sex, along with pregnancy-related deaths segmented by age and occupation of the deceased mother, across urban and rural districts.",
  "last_updated": null,
  "future_release": null,
  "key_statistics": {
    "total_infant_deaths": 6866,
    "total_grave_urban_deaths": 5225,
    "total_grave_rural_deaths": 1641,
    "total_mother_deaths_urban": 58,
    "total_mother_deaths_rural": 55
  },
  "remarks": "Data is presented at the district level and segregated into urban and rural areas. Infant deaths are further detailed by age and sex, while maternal deaths are categorized by age and occupation."
}
```


### Parse the LLM output and display the filled metadata fields

`metadata_llm.parse_llm_metadata_output` strips any markdown fencing and parses the JSON returned
above into a plain dict normalized to the catalogue's metadata fields (missing fields become `None`).

In [11]:
from metadata_llm import parse_llm_metadata_output

if not SKIP_LLM:
    llm_filled_metadata = parse_llm_metadata_output(llm_metadata)
else:
    print("SKIP_LLM=True — no LLM output to parse. Showing the Stage 3 catalogue_summary prefill instead:")
    llm_filled_metadata = {field: prefill_fields.get(field) for field in
                            ["title", "product", "category", "geography", "frequency", "time_period",
                             "data_source", "description", "last_updated", "future_release",
                             "key_statistics", "remarks"]}

print("LLM-filled metadata fields:")
for field, value in llm_filled_metadata.items():
    print(f"  {field:16}: {value}")

LLM-filled metadata fields:
  title           : Still Births by Place of Occurrence in Districts
  product         : Still Birth Data
  category        : Health
  geography       : ['District', 'Urban', 'Rural']
  frequency       : Annual
  time_period     : None
  data_source     : None
  description     : Dataset detailing stillbirth occurrences by gender and place of occurrence (institutional or domiciliary) in urban and rural districts.
  last_updated    : None
  future_release  : None
  key_statistics  : {'total_urban_stillbirths': 1645, 'total_rural_stillbirths': 560}
  remarks         : None


---
## Summary — Simplified End-to-End Pipeline

The notebook now demonstrates an **automatic, hardcode-free pipeline**:

1. ✅ **Extract tables** from workbook (Stage 1)
2. ✅ **KYDS form** submission (Stage 2)
3. ✅ **Auto-group tables** by ID patterns (URBAN/RURAL pairs, same sheet) (Stage 3)
4. ✅ **LLM metadata creation** per group from excel_facts + KYDS (Stage 4)
5. ✅ **Metadata filling & push** to catalogue (Stage 5)

**Key differences from old flow:**
- ❌ No hardcoded `build_sample_metadata_workbook()` function
- ❌ No external metadata workbook required
- ✅ Grouping is automatic, based on table ID patterns
- ✅ Metadata is generated once per group (not per table)

⏸️ **Next stage** (not in this notebook): per-table **column classification**
(`TableExtractor.enrich_for_catalogue`) to populate `units`, `classifications`, and `age_column_keys`.